# Oracle PL/SQL Engines: Autonomous Transactions & Compound Triggers

Interactive hands-on sandbox exploring internal storage mechanics, algorithmic data structures, and architectural invariants.


In [ ]:
import sys
from pathlib import Path

# Prepend project_solution to sys.path so Track A internal engines import cleanly
ps_dir = Path('.').resolve() / 'project_solution'
if str(ps_dir) not in sys.path:
    sys.path.insert(0, str(ps_dir))

print(f'Python runtime: {sys.version.split()[0]}')
print('Loaded internal mechanics for: Module_08_Oracle_PLSQL_Packages_Triggers')


## 1. Engine Initialization & Setup

Importing the module's Track A internal simulation engine and instantiating state.


In [ ]:
from oracle_plsql_engine import AutonomousAuditLogger, BankingPackage

# Initialize Autonomous Audit Logger and Banking Package
audit_logger = AutonomousAuditLogger()
bank = BankingPackage(audit_logger)

# Open accounts
bank.create_account("ACC_100", 5000.0)
bank.create_account("ACC_200", 2000.0)
print(f"Accounts created. Active ledger: {bank.accounts}")


## 2. Core Architectural Operations & State Mutation

Executing data mutations, transactions, or indexing procedures.


In [ ]:
# Successful Transfer with PL/SQL Transaction Boundary
transfer_ok = bank.transfer_funds("ACC_100", "ACC_200", 1500.0)
print(f"Transfer succeeded: {transfer_ok}")
print(f"Balances post-transfer: ACC_100=${bank.accounts['ACC_100']}, ACC_200=${bank.accounts['ACC_200']}")
print(f"Committed audit log entries: {len(audit_logger.committed_logs)}")


## 3. Performance Micro-Benchmarking & Invariant Verification

Evaluating execution latency, cache hits, or computational trade-offs.


In [ ]:
# PRAGMA AUTONOMOUS_TRANSACTION Mechanics:
# When parent transaction aborts/rolls back, the autonomous audit log MUST PERSIST!
try:
    # Insufficient funds triggers exception
    bank.transfer_funds("ACC_200", "ACC_100", 999999.0)
except ValueError as e:
    print("Caught expected transaction exception:", e)

print(f"Audit log size after failed transfer: {len(audit_logger.committed_logs)} (Audit preserved!)")
for log in audit_logger.committed_logs:
    print(f"  Log entry: id={log.event_id}, event_type={log.event_type}, details={log.details}")


## 4. Architectural Invariant Verification

Asserting mathematical correctness and durability invariants.


In [ ]:
# Verify PL/SQL Invariants
assert bank.accounts["ACC_100"] == 3500.0
assert bank.accounts["ACC_200"] == 3500.0
assert any(log.event_type == "INSUFFICIENT_FUNDS" for log in audit_logger.committed_logs)
print("[+] Oracle PL/SQL Autonomous Transaction & Trigger invariants verified successfully!")


## Summary & Operational DBRE Best Practices

1. **Never bypass serialization contracts:** Always enforce binary-safe schemas and validated boundaries.
2. **Monitor buffer and memory allocations:** Understand the latency cliff when in-memory structures spill to disk.
3. **Ensure idempotency across replication tiers:** Distributed operations must survive retries without corrupting state.
